# PathSnap — Member 2: Model Training & Candidate Probabilities

This notebook trains and compares five ML models using the preprocessed PathSnap candidate-road dataset, tunes the best model with Optuna, saves the final model, and exports candidate probabilities for the HMM stage.

## How to run
1. Open this notebook in Google Colab.
2. Run the single code cell.
3. When prompted, upload `pathsnap_preprocessed_outputs.zip` from the preprocessing step.
4. Wait for training and download `pathsnap_model_training_outputs.zip`.

## Models
- Logistic Regression
- K-Nearest Neighbours (KNN)
- Random Forest
- Extra Trees
- XGBoost

## Important design choices
- The train/validation split is grouped by `trip_id` to avoid putting observations from the same trip in both sets.
- Validation ranking is measured by **top-1 candidate accuracy**: for each GPS observation, choose the candidate arc with the highest ML probability and compare it with the supplied label.
- Large-data safeguards sample complete observation groups for the main experiments and use row caps for expensive estimators. You can increase the limits in the CONFIG section if Colab has enough RAM/time.
- The student-test partition is not used for model selection or tuning. It is only scored after the final model is fitted on labelled training data.
- `ml_probability` is the classifier's positive-class probability for each candidate. Values are **not normalized to sum to 1** across candidates. No candidates are filtered by probability.
- Output metadata (`trip_id`, `observation_id`, `timestamp`, `arc_id`) is kept separate from ML features and preserved for Member 3.

## Important label caveat
The supplied preprocessing output creates a positive label by adding the nearest reference-route arc to each training observation's candidate set. This is a heuristic label-generation method. Before treating metrics as final project results, confirm that this label definition is consistent with the project guidelines and reference-route format. The notebook reports validation performance on these generated labels, not ground-truth road-sequence accuracy.

## Outputs
- `pathsnap_best_model.joblib`
- `pathsnap_model_comparison.csv`
- `pathsnap_optuna_trials.csv`
- `pathsnap_candidate_probabilities.csv`
- `pathsnap_hmm_input.json` — nested structure for Member 3
- `pathsnap_final_metrics.json`


In [ ]:

# ====================== PATHSNAP MEMBER 2: MODEL TRAINING ======================
# Run this ONE cell in Google Colab.
# Input: pathsnap_preprocessed_outputs.zip (downloaded from the preprocessing notebook)
# Output: trained model + comparison + all student-test candidate probabilities + HMM JSON.

import os, sys, subprocess, importlib.util, zipfile, json, warnings, time
from pathlib import Path

def ensure_package(package, import_name=None):
    if importlib.util.find_spec(import_name or package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package])

ensure_package("xgboost")
ensure_package("optuna")
ensure_package("joblib")

import numpy as np
import pandas as pd
import joblib
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from google.colab import files

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ----------------------------- CONFIG -------------------------------------
SEED = 42
MAX_COMPARE_TRAIN_OBSERVATIONS = 5000   # complete GPS observations, not random candidate rows
MAX_VALIDATION_OBSERVATIONS = 1000
MAX_TUNING_TRAIN_OBSERVATIONS = 5000
MAX_FINAL_TRAIN_OBSERVATIONS = 30000
OPTUNA_TRIALS = 10
OUTPUT_DIR = "/content/pathsnap_model_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)
FEATURES = [
    "distance_m", "heading_diff_deg", "speed_mps",
    "relative_distance_m", "connectivity"
]
METADATA = ["trip_id", "observation_id", "timestamp", "arc_id"]
LABEL = "label"

# ------------------------- UPLOAD / EXTRACT -------------------------------
print("Upload pathsnap_preprocessed_outputs.zip")
uploaded = files.upload()
if not uploaded:
    raise RuntimeError("No ZIP uploaded.")
zip_name = next(iter(uploaded))
zip_path = f"/content/{zip_name}"
with open(zip_path, "wb") as f:
    f.write(uploaded[zip_name])

data_dir = "/content/pathsnap_model_data"
if os.path.exists(data_dir):
    import shutil
    shutil.rmtree(data_dir)
os.makedirs(data_dir, exist_ok=True)
with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(data_dir)

train_path = os.path.join(data_dir, "pathsnap_train_candidates.csv")
test_path = os.path.join(data_dir, "pathsnap_student_test_candidates.csv")
if not os.path.exists(train_path) or not os.path.exists(test_path):
    raise FileNotFoundError("ZIP must contain pathsnap_train_candidates.csv and pathsnap_student_test_candidates.csv")

usecols_train = METADATA + FEATURES + [LABEL]
usecols_test = METADATA + FEATURES
dtype_map = {"trip_id": "string"}
print("Loading training CSV (large file; this may take a little while)...")
train_df = pd.read_csv(train_path, usecols=usecols_train, dtype=dtype_map)
print("Loading student-test CSV...")
test_df = pd.read_csv(test_path, usecols=usecols_test, dtype=dtype_map)

# Metadata remains separate from model features. Never feed IDs/timestamps to the classifier.
for col in FEATURES:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce")
train_df[LABEL] = pd.to_numeric(train_df[LABEL], errors="coerce").fillna(0).astype("int8")
train_df["observation_id"] = pd.to_numeric(train_df["observation_id"], errors="coerce").astype("int64")
test_df["observation_id"] = pd.to_numeric(test_df["observation_id"], errors="coerce").astype("int64")
train_df["arc_id"] = pd.to_numeric(train_df["arc_id"], errors="coerce").astype("int64")
test_df["arc_id"] = pd.to_numeric(test_df["arc_id"], errors="coerce").astype("int64")

print(f"Training candidate rows: {len(train_df):,}")
print(f"Student-test candidate rows: {len(test_df):,}")
print(f"Training trips: {train_df.trip_id.nunique()} | Student-test trips: {test_df.trip_id.nunique()}")
print(f"Positive label rows: {int(train_df[LABEL].sum()):,} ({train_df[LABEL].mean():.2%})")

# Check whether each labelled observation has exactly one positive candidate.
label_counts = train_df.groupby(["trip_id", "observation_id"], sort=False)[LABEL].sum()
print("Observations with exactly one positive candidate:",
      f"{int((label_counts == 1).sum()):,} / {len(label_counts):,}")
if not (label_counts == 1).all():
    print("WARNING: Some observations do not have exactly one positive candidate. Review preprocessing labels.")

# --------------------- TRIP-GROUPED TRAIN/VALIDATION SPLIT -----------------
unique_trips = train_df["trip_id"].drop_duplicates().to_numpy()
splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
train_trip_idx, val_trip_idx = next(splitter.split(unique_trips, groups=unique_trips))
fit_trip_ids = set(unique_trips[train_trip_idx])
val_trip_ids = set(unique_trips[val_trip_idx])

fit_pool = train_df[train_df.trip_id.isin(fit_trip_ids)].copy()
val_pool = train_df[train_df.trip_id.isin(val_trip_ids)].copy()
print(f"Trip split: {len(fit_trip_ids)} fit trips, {len(val_trip_ids)} validation trips")

def sample_complete_observations(df, max_observations, seed):
    """Sample observation groups while retaining every candidate for each selected observation."""
    keys = df[["trip_id", "observation_id"]].drop_duplicates()
    if len(keys) <= max_observations:
        return df.copy()
    chosen = keys.sample(n=max_observations, random_state=seed)
    chosen = chosen.assign(_keep=1)
    sampled = df.merge(chosen, on=["trip_id", "observation_id"], how="inner", sort=False)
    return sampled.drop(columns=["_keep"])

def cap_rows_stratified(df, max_rows, seed):
    """Optional row cap for computationally heavy estimators; retains natural class balance."""
    if len(df) <= max_rows:
        return df
    idx = np.arange(len(df))
    keep_idx, _ = train_test_split(
        idx, train_size=max_rows, random_state=seed,
        stratify=df[LABEL].to_numpy()
    )
    return df.iloc[keep_idx].copy()

fit_sample = sample_complete_observations(
    fit_pool, MAX_COMPARE_TRAIN_OBSERVATIONS, SEED
)
val_sample = sample_complete_observations(
    val_pool, MAX_VALIDATION_OBSERVATIONS, SEED + 1
)
print(f"Model-comparison training sample: {len(fit_sample):,} candidate rows")
print(f"Validation sample: {len(val_sample):,} candidate rows")

# --------------------------- MODEL BUILDERS -------------------------------
def make_model(name, params=None):
    p = params or {}
    if name == "Logistic Regression":
        clf = LogisticRegression(
            C=p.get("C", 1.0), max_iter=500, class_weight="balanced",
            solver="lbfgs", n_jobs=None
        )
        return Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("scaler", StandardScaler()),
                         ("classifier", clf)])
    if name == "KNN":
        clf = KNeighborsClassifier(
            n_neighbors=p.get("n_neighbors", 7),
            weights=p.get("weights", "distance"),
            algorithm="kd_tree", leaf_size=40, n_jobs=-1
        )
        return Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("scaler", StandardScaler()),
                         ("classifier", clf)])
    if name == "Random Forest":
        clf = RandomForestClassifier(
            n_estimators=p.get("n_estimators", 150),
            max_depth=p.get("max_depth", 18),
            min_samples_leaf=p.get("min_samples_leaf", 2),
            max_features=p.get("max_features", "sqrt"),
            class_weight="balanced_subsample",
            random_state=SEED, n_jobs=-1
        )
        return Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("classifier", clf)])
    if name == "Extra Trees":
        clf = ExtraTreesClassifier(
            n_estimators=p.get("n_estimators", 150),
            max_depth=p.get("max_depth", 20),
            min_samples_leaf=p.get("min_samples_leaf", 2),
            max_features=p.get("max_features", "sqrt"),
            class_weight="balanced",
            random_state=SEED, n_jobs=-1
        )
        return Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("classifier", clf)])
    if name == "XGBoost":
        clf = XGBClassifier(
            n_estimators=p.get("n_estimators", 250),
            max_depth=p.get("max_depth", 6),
            learning_rate=p.get("learning_rate", 0.08),
            subsample=p.get("subsample", 0.85),
            colsample_bytree=p.get("colsample_bytree", 0.9),
            min_child_weight=p.get("min_child_weight", 3),
            objective="binary:logistic", eval_metric="logloss",
            tree_method="hist", n_jobs=-1, random_state=SEED,
            scale_pos_weight=p.get("scale_pos_weight", 1.0)
        )
        return Pipeline([("imputer", SimpleImputer(strategy="median")),
                         ("classifier", clf)])
    raise ValueError("Unknown model: " + name)

def positive_probability(model, frame):
    return model.predict_proba(frame[FEATURES])[:, 1]

def evaluate_candidate_ranking(frame, probabilities):
    """Top-1 accuracy selects the highest-probability arc for each GPS observation."""
    tmp = frame[["trip_id", "observation_id", LABEL]].copy()
    tmp["probability"] = probabilities
    chosen_idx = tmp.groupby(["trip_id", "observation_id"], sort=False)["probability"].idxmax()
    top1 = float(tmp.loc[chosen_idx, LABEL].mean())
    pred = (probabilities >= 0.5).astype(int)
    y = frame[LABEL].to_numpy()
    try:
        auc = float(roc_auc_score(y, probabilities))
    except Exception:
        auc = float("nan")
    return {
        "top1_accuracy": top1,
        "binary_f1_at_0.5": float(f1_score(y, pred, zero_division=0)),
        "precision_at_0.5": float(precision_score(y, pred, zero_division=0)),
        "recall_at_0.5": float(recall_score(y, pred, zero_division=0)),
        "roc_auc": auc
    }

model_names = ["Logistic Regression", "KNN", "Random Forest", "Extra Trees", "XGBoost"]
comparison_rows = []
comparison_models = {}

for name in model_names:
    print(f"\\nTraining {name}...")
    # KNN is deliberately capped more tightly because neighbour search is expensive.
    model_train = cap_rows_stratified(
        fit_sample, 25000 if name == "KNN" else 100000, SEED
    )
    model = make_model(name)
    start = time.time()
    model.fit(model_train[FEATURES], model_train[LABEL])
    probs = positive_probability(model, val_sample)
    metrics = evaluate_candidate_ranking(val_sample, probs)
    metrics.update({
        "model": name,
        "fit_rows": len(model_train),
        "validation_observations": val_sample[["trip_id", "observation_id"]].drop_duplicates().shape[0],
        "fit_seconds": round(time.time() - start, 1)
    })
    comparison_rows.append(metrics)
    comparison_models[name] = model
    print(metrics)

comparison = pd.DataFrame(comparison_rows).sort_values(
    "top1_accuracy", ascending=False
).reset_index(drop=True)
comparison_path = os.path.join(OUTPUT_DIR, "pathsnap_model_comparison.csv")
comparison.to_csv(comparison_path, index=False)
print("\\nMODEL COMPARISON")
display(comparison)

best_name = str(comparison.iloc[0]["model"])
print("Selected model based on validation top-1 accuracy:", best_name)

# ----------------------------- OPTUNA --------------------------------------
tune_train = sample_complete_observations(
    fit_pool, MAX_TUNING_TRAIN_OBSERVATIONS, SEED + 2
)
tune_train = cap_rows_stratified(
    tune_train, 25000 if best_name == "KNN" else 100000, SEED + 3
)
print(f"Optuna tuning rows: {len(tune_train):,}; trials: {OPTUNA_TRIALS}")

def objective(trial):
    if best_name == "Logistic Regression":
        params = {"C": trial.suggest_float("C", 0.01, 20.0, log=True)}
    elif best_name == "KNN":
        params = {
            "n_neighbors": trial.suggest_int("n_neighbors", 3, 15, step=2),
            "weights": trial.suggest_categorical("weights", ["uniform", "distance"])
        }
    elif best_name in ("Random Forest", "Extra Trees"):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 220, step=40),
            "max_depth": trial.suggest_int("max_depth", 8, 28, step=4),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 5),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", 0.8])
        }
    else:
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 150, 350, step=50),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
            "subsample": trial.suggest_float("subsample", 0.7, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.7, 1.0),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 8)
        }
    model = make_model(best_name, params)
    model.fit(tune_train[FEATURES], tune_train[LABEL])
    probs = positive_probability(model, val_sample)
    score = evaluate_candidate_ranking(val_sample, probs)["top1_accuracy"]
    return score

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=SEED)
)
study.optimize(objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)
best_params = study.best_params
print("Best Optuna top-1 accuracy:", study.best_value)
print("Best parameters:", best_params)

trials_path = os.path.join(OUTPUT_DIR, "pathsnap_optuna_trials.csv")
study.trials_dataframe().to_csv(trials_path, index=False)

# ------------------------- FINAL MODEL FIT --------------------------------
# Refit selected/tuned estimator using labelled training trips (including validation trips).
# We retain complete observation groups when sampling the large dataset.
final_train = sample_complete_observations(
    train_df, MAX_FINAL_TRAIN_OBSERVATIONS, SEED + 4
)
# Final cap is a compute safeguard. For KNN, keep a smaller reference set.
final_cap = 30000 if best_name == "KNN" else 500000
final_train = cap_rows_stratified(final_train, final_cap, SEED + 5)

final_model = make_model(best_name, best_params)
print(f"\\nFitting final {best_name} model on {len(final_train):,} candidate rows...")
final_model.fit(final_train[FEATURES], final_train[LABEL])

# Store feature order and metadata contract with the model.
model_bundle = {
    "model": final_model,
    "model_name": best_name,
    "best_params": best_params,
    "feature_columns": FEATURES,
    "metadata_columns": METADATA,
    "positive_class": 1,
    "probability_definition": "predict_proba[:, 1]; independent per-candidate score, not normalized across candidates"
}
model_path = os.path.join(OUTPUT_DIR, "pathsnap_best_model.joblib")
joblib.dump(model_bundle, model_path)

# --------------------- PREDICT ALL STUDENT-TEST CANDIDATES ----------------
# Do NOT filter low-probability candidates and do NOT normalize probabilities to sum to one.
def predict_candidates(feature_rows, bundle):
    """Classifier-independent interface for Member 3: returns every candidate row."""
    model = bundle["model"]
    feature_cols = bundle["feature_columns"]
    result = feature_rows[METADATA].copy()
    result["ml_probability"] = model.predict_proba(
        feature_rows[feature_cols]
    )[:, 1]
    return result

print("\\nPredicting every student-test candidate row...")
prediction_parts = []
batch_size = 50000
for start in range(0, len(test_df), batch_size):
    batch = test_df.iloc[start:start + batch_size]
    prediction_parts.append(predict_candidates(batch, model_bundle))
    print(f"Predicted {min(start + batch_size, len(test_df)):,}/{len(test_df):,}")

predictions = pd.concat(prediction_parts, ignore_index=True)
pred_csv = os.path.join(OUTPUT_DIR, "pathsnap_candidate_probabilities.csv")
predictions.to_csv(pred_csv, index=False)

# Nested JSON contract for Member 3 / HMM. Input row order is preserved.
# Every candidate is retained, including very low probability candidates.
json_path = os.path.join(OUTPUT_DIR, "pathsnap_hmm_input.json")
nested = []
for trip_id, trip_group in predictions.groupby("trip_id", sort=False):
    observations = []
    for obs_id, obs_group in trip_group.groupby("observation_id", sort=False):
        first = obs_group.iloc[0]
        candidates = [
            {"arc_id": int(row.arc_id), "ml_probability": float(row.ml_probability)}
            for row in obs_group.itertuples(index=False)
        ]
        observations.append({
            "observation_id": int(obs_id),
            "timestamp": float(first.timestamp),
            "candidates": candidates
        })
    nested.append({"trip_id": str(trip_id), "observations": observations})

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(nested, f, separators=(",", ":"))

# ----------------------------- SAVE METRICS --------------------------------
best_comparison_row = comparison[comparison.model == best_name].iloc[0].to_dict()
final_info = {
    "selected_model": best_name,
    "best_optuna_parameters": best_params,
    "validation_top1_accuracy_before_tuning": float(best_comparison_row["top1_accuracy"]),
    "optuna_best_validation_top1_accuracy": float(study.best_value),
    "training_candidate_rows_used_for_final_fit": int(len(final_train)),
    "student_test_candidate_rows_predicted": int(len(predictions)),
    "student_test_trips": int(predictions.trip_id.nunique()),
    "probability_output": "positive class probability for every candidate; not sum-normalized",
    "feature_columns": FEATURES,
    "metadata_columns": METADATA,
    "label_caveat": "Labels come from the preprocessing notebook's nearest reference-route arc heuristic; validate this against the project specification before reporting as ground truth."
}
metrics_path = os.path.join(OUTPUT_DIR, "pathsnap_final_metrics.json")
with open(metrics_path, "w") as f:
    json.dump(final_info, f, indent=2)

# Zip deliverables.
output_zip = "/content/pathsnap_model_training_outputs.zip"
with zipfile.ZipFile(output_zip, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in [model_path, comparison_path, trials_path, pred_csv, json_path, metrics_path]:
        zf.write(p, arcname=os.path.basename(p))

print("\\n================ TRAINING COMPLETE ================")
print("Selected model:", best_name)
print("Student-test candidate probabilities:", len(predictions))
print("HMM JSON:", json_path)
print("Saved model:", model_path)
print("Download bundle:", output_zip)
print("\\nReminder: student-test trips have no labels in this workflow, so their accuracy is not reported.")
files.download(output_zip)
# =============================================================================
